# Held-out exact-integer base gate v1

Stopped. The mlx run at `5d36336ad0272d59ca815646fcb2f812849be1c5` is `stop_zero`. Do not run these cells again, and do not edit the pack or parser. v2 is a separate protocol.

GPU runtime. The decoder is one sample at a time from `HuggingFaceTB/SmolLM2-360M-Instruct` in float16 when CUDA is available, with at most 48 new tokens. A T4, L4, or A100 uses that same decoder.

Run the cells in order. The gate appends to `samples.jsonl` and skips completed samples. Put the output on Drive so a disconnect can be resumed. `summary.json` appears only after 256 graded samples.

The last cell runs `scripts/heldout_exact_integer_update_comparison_v1.py`. That script refuses a best-of-n versus GRPO or RFT comparison and exits 2. Do not add a training cell to this notebook.


In [ ]:
import os
from pathlib import Path

os.environ.setdefault("VARE_REPO", "https://github.com/mitukx/VARE.git")
os.environ.setdefault("VARE_REV", "cursor/heldout-exact-integer-gate-af75")
MOUNT_DRIVE = True
REPO = Path("/content/vare-heldout-exact-integer-v1")
print(os.environ["VARE_REPO"], os.environ["VARE_REV"])


In [ ]:
import os
import subprocess
from pathlib import Path

repo = Path("/content/vare-heldout-exact-integer-v1")
if not (repo / ".git").exists():
    subprocess.check_call(["git", "clone", os.environ["VARE_REPO"], str(repo)])
subprocess.check_call(["git", "fetch", "origin", os.environ["VARE_REV"]], cwd=repo)
subprocess.check_call(["git", "checkout", "--detach", "FETCH_HEAD"], cwd=repo)
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
print(revision)


In [ ]:
import subprocess
import sys

subprocess.check_call(
    [sys.executable, "-m", "pip", "install", "-q", "-e", "/content/vare-heldout-exact-integer-v1", "transformers", "accelerate", "huggingface_hub"]
)


In [ ]:
import os
from pathlib import Path

output = Path("/content/heldout-exact-integer-base-gate-v1")
if MOUNT_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    root = Path("/content/drive/MyDrive/vare")
    root.mkdir(parents=True, exist_ok=True)
    output = root / "heldout-exact-integer-base-gate-v1"
    os.environ["HF_HOME"] = str(root / "huggingface-cache")
    os.environ["HF_HUB_CACHE"] = str(root / "huggingface-cache" / "hub")
output.mkdir(parents=True, exist_ok=True)
print("output", output)
print("HF_HOME", os.environ.get("HF_HOME", "(runtime default)"))
if not str(output).startswith("/content/drive/"):
    print("Drive is not mounted. A runtime reset deletes samples under /content.")


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo = Path("/content/vare-heldout-exact-integer-v1")
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=repo, text=True).strip()
(output / "code_revision.txt").write_text(revision + "\n", encoding="utf-8")
env = os.environ.copy()
env["PYTHONPATH"] = str(repo / "src") + os.pathsep + env.get("PYTHONPATH", "")
command = [
    sys.executable,
    str(repo / "scripts/run_heldout_exact_integer_base_gate_v1.py"),
    "--backend",
    "transformers",
    "--output",
    str(output),
]
print(" ".join(command))
completed = subprocess.call(command, cwd=repo, env=env)
if completed != 0:
    raise RuntimeError(f"base gate exited {completed}")


In [ ]:
import json

summary_path = output / "summary.json"
if summary_path.exists():
    print(summary_path.read_text(encoding="utf-8"))
else:
    samples = output / "samples.jsonl"
    count = 0 if not samples.exists() else sum(1 for line in samples.read_text(encoding="utf-8").splitlines() if line.strip())
    print(json.dumps({"complete": False, "graded_samples": count, "expected_samples": 256}))


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

repo = Path("/content/vare-heldout-exact-integer-v1")
env = os.environ.copy()
env["PYTHONPATH"] = str(repo / "src") + os.pathsep + env.get("PYTHONPATH", "")
proc = subprocess.run(
    [sys.executable, str(repo / "scripts/heldout_exact_integer_update_comparison_v1.py")],
    cwd=repo,
    env=env,
    capture_output=True,
    text=True,
    check=False,
)
print(proc.stdout)
print("comparison_exit", proc.returncode)
if proc.returncode != 2:
    raise RuntimeError("the update comparison must exit 2")
print("Stopped. No policy-update comparison was started.")
